In [ ]:
# 再実行時: 残った ngrok を終了（トンネル重複・URL取り違い防止）
import time

try:
    from pyngrok import ngrok

    ngrok.kill()
    time.sleep(2)
except Exception:
    pass

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

%cd /content
!rm -rf aibo_v8
!git clone -b colab-stable https://github.com/miya390831-a11y/aibo_v8.git
%cd /content/aibo_v8

# 全パッケージインストール（os.kill なし）
!pip install -q --upgrade nunchaku
!pip install -q --upgrade --force-reinstall "diffusers>=0.36"
!pip install -q "transformers>=4.54" "accelerate>=1.9" "peft>=0.17"
!pip install -q "huggingface_hub>=0.34" "scipy>=1.14"
!pip install -q pyngrok

# importlib をリロード（os.kill 不要）
import importlib, site

importlib.reload(site)

print('✅ Setup 完了')

In [ ]:
%cd /content/aibo_v8
import os
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HUGGINGFACE_HUB_TOKEN'] = userdata.get('HF_TOKEN')

import importlib.util, sys

sys.path.insert(0, '/content/aibo_v8')


def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod


cfg_mod = load_module('01_config', '01_config.py')
setup_mod = load_module('02_colab_setup', '02_colab_setup.py')
sys_cfg = cfg_mod.SystemConfig()
setup_mod.ColabBootstrap(sys_cfg).run()
print('✅ Bootstrap 完了')

In [ ]:
%cd /content/aibo_v8
main_mod = load_module('07_main', '07_main.py')
main_mod.run()
print('✅ Main 起動完了')

In [ ]:
%cd /content/aibo_v8
import subprocess, time, os
from pyngrok import ngrok, conf
from google.colab import userdata

conf.get_default().auth_token = userdata.get("NGROK_TOKEN")
ngrok.kill()
time.sleep(2)

# FastAPI 起動（localhost のみ · ブラウザは Next がプロキシ）
fastapi_proc = subprocess.Popen(
    ["python", "09_fastapi_server.py"],
    cwd="/content/aibo_v8",
)
time.sleep(10)
print("✅ FastAPI 起動（localhost:8000）")

FE = "/content/aibo_v8/frontend"
# 同じオリジンで /api → Route Handler が 8000 へ中継
next_env = os.environ.copy()
next_env["NEXT_PUBLIC_API_URL"] = ""

%cd /content/aibo_v8/frontend
!rm -rf .next
!npm install --silent
subprocess.run(["npm", "run", "build"], cwd=FE, env=next_env, check=True)

nextjs_proc = subprocess.Popen(["npm", "start"], cwd=FE, env=next_env)
time.sleep(15)

# ngrok は UI（3000）だけ
ui_tunnel = ngrok.connect(3000, "http")
ui_url_str = ui_tunnel.public_url

print(f"🌐 UI URL: {ui_url_str}")